# AISO Few-Shot Selector — 승정원일기 번역

## 목적
현재 `fewshot_config.json`의 고정 5예시는 **전부 REP 패턴(아뢰기/전교)**입니다.  
AISO는 패턴 타입·길이·개체명 밀도가 겹치지 않는 예시 조합을 자동으로 찾습니다.

## 두 가지 선택 모드
| 모드 | 설명 | 사용처 |
|------|------|--------|
| **Global** | 1회 실행 → 모든 쿼리에 같은 예시 | `fewshot_config.json` 교체 |
| **Per-query** | 쿼리별로 패턴 매칭 + 다양성 선택 | `run_fewshot.py` 통합 |

In [ ]:
import json, re, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from pathlib import Path

BASE = Path(".")  # ablation5way/ 기준
np.random.seed(42)
random.seed(42)

---
## 1. 데이터 로드 및 풀 구성

eval300 ID를 제외한 62K 코퍼스 전체가 풀입니다.

In [ ]:
# eval300 ID 로드 (풀에서 제외)
with open(BASE / "eval300_1925.json", "r", encoding="utf-8") as f:
    eval300 = json.load(f)
eval_ids = set(eval300["ids"])
print(f"eval300 제외 ID 수: {len(eval_ids)}")

# 62K 코퍼스 로드
print("Merged_Corpus_Final.json 로딩 중...", end=" ")
with open(BASE / "Merged_Corpus_Final.json", "r", encoding="utf-8") as f:
    corpus_raw = json.load(f)
corpus = corpus_raw["corpus"]
print(f"완료. 전체 항목: {len(corpus):,}개")

# eval300 제외, 번역문 있는 항목만
pool = [
    e for e in corpus
    if e["id"] not in eval_ids
    and e.get("translation", "").strip()
    and e.get("original", "").strip()
]
print(f"풀 크기: {len(pool):,}개 (eval300 제외, 번역 존재)")

---
## 2. 도메인 피처 설계

보고서의 패턴 분류(REP/GEN/DEN/MEM/ROY)와 길이 버킷(XS/S/M/L)을 피처로 직접 인코딩합니다.

| 피처 | 설명 | 캡처하는 역할 |
|------|------|---------------|
| `length_norm` | len(original) / 350, clip 1.0 | XS~L 길이 스펙트럼 |
| `has_kye` | 啓曰/又啓曰 포함 여부 | REP 패턴(아뢰기) |
| `has_den` | 不許/不允 포함 여부 | DEN 패턴(불허) |
| `has_mem` | 狀啓/書啓 포함 여부 | MEM 패턴(장계/서계) |
| `has_roy` | 上曰/傳曰 포함 여부 | ROY 패턴(왕 발화) |
| `entity_density` | 한자 고유명사 밀도 추정 | 인물/지명 밀집도 |
| `date_density` | 月/日 토큰 밀도 | 날짜 표현 집중도 |
| `vocab_richness` | 고유 한자 / 전체 한자 | 어휘 다양성 |

In [ ]:
_RE_HANJA = re.compile(r'[一-鿿]')

def extract_features(entry: dict) -> np.ndarray:
    """한문 원문(original)에서 8차원 도메인 피처 추출"""
    orig = entry["original"]
    n = max(len(orig), 1)
    hanja = _RE_HANJA.findall(orig)
    n_hanja = max(len(hanja), 1)

    length_norm   = min(n / 350.0, 1.0)
    has_kye       = float('啓曰' in orig or '又啓' in orig)
    has_den       = float('不許' in orig or '不允' in orig)
    has_mem       = float('狀啓' in orig or '書啓' in orig)
    has_roy       = float('上曰' in orig or '傳曰' in orig or '上이' in orig)
    # ·는 승정원일기에서 인명 나열 시 쓰는 구분자 (金瑬·李貴·沈器遠)
    entity_density   = min(orig.count('·') / n * 5, 1.0)
    date_density     = min((orig.count('月') + orig.count('日')) / n * 10, 1.0)
    unique_hanja     = len(set(hanja))
    vocab_richness   = unique_hanja / n_hanja

    return np.array([
        length_norm,
        has_kye,
        has_den,
        has_mem,
        has_roy,
        entity_density,
        date_density,
        vocab_richness,
    ], dtype=float)

FEATURE_NAMES = [
    "length_norm", "has_kye(REP)", "has_den(DEN)",
    "has_mem(MEM)", "has_roy(ROY)", "entity_density",
    "date_density", "vocab_richness"
]

In [ ]:
# 풀 전체 피처 추출 (전체 또는 샘플)
# 전체 62K 사용 시 약 10~30초 소요
USE_FULL_POOL = True   # False로 바꾸면 5000개 샘플로 빠른 테스트

if USE_FULL_POOL:
    work_pool = pool
else:
    work_pool = random.sample(pool, min(5000, len(pool)))

print(f"피처 추출 중 ({len(work_pool):,}개)...", end=" ")
X_pool = np.array([extract_features(e) for e in work_pool])
print("완료")
print(f"피처 행렬: {X_pool.shape}")

# 피처 분포 확인
feat_df = pd.DataFrame(X_pool, columns=FEATURE_NAMES)
print("\n피처 통계:")
print(feat_df.describe().round(3).to_string())

In [ ]:
# 패턴 분포 확인
pattern_labels = []
for feat in X_pool:
    if feat[1] > 0:   pattern_labels.append("REP(啓曰)")
    elif feat[2] > 0: pattern_labels.append("DEN(不許)")
    elif feat[3] > 0: pattern_labels.append("MEM(狀啓)")
    elif feat[4] > 0: pattern_labels.append("ROY(上曰)")
    else:             pattern_labels.append("GEN(일반)")

from collections import Counter
cnt = Counter(pattern_labels)
print("풀 내 패턴 분포:")
for k, v in sorted(cnt.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v:,}개 ({v/len(pattern_labels):.1%})")

---
## 3. AISO 구현

### 작동 원리 (3줄 요약)
1. **에이전트 N개**가 피처 공간을 돌아다니며 가장 가까운 풀 아이템을 방문 (`visit` +1)
2. 에이전트끼리 $c_{ij} = W_i^\top M W_j$ 값으로 끌리거나 밀림 → 서로 다른 피처 영역 탐색
3. 반복 후 **많이 방문된 아이템 = 피처 공간을 고르게 대표하는 예시**

### 왜 nearest-neighbor 스냅인가?
에이전트가 임의 위치로 이동하면 실제 존재하지 않는 예시가 됩니다.  
스냅은 에이전트가 항상 실제 풀 아이템 위에 머물도록 강제합니다.

### visit_counts의 의미
많이 방문된 아이템일수록 여러 에이전트가 그 주변을 탐색했다는 의미 →  
피처 공간에서 구조적으로 대표적인 위치에 있는 아이템입니다.  
최종 선택은 이 빈도에 비례한 확률 샘플링으로 이루어집니다.

In [ ]:
class AISOfewshot:
    """
    AISO 기반 Few-Shot 예시 선택기.

    파라미터
    --------
    n_agents : 에이전트 수. 권장 15~30
    n_iter   : 반복 횟수. 권장 80~150
    n_types  : 타입 벡터 차원 K. None이면 피처 차원과 동일
    alpha    : 이동 스텝 크기. 권장 0.2~0.3
    beta     : 타입 동화율. 권장 0.05~0.10
    m_low    : M 행렬 하한 (음수 → repulsion). 권장 -0.5~-1.0
    m_high   : M 행렬 상한. 권장 1.5~2.5
    """
    def __init__(self, n_agents=20, n_iter=100, n_types=None,
                 alpha=0.25, beta=0.08, m_low=-0.5, m_high=2.0, seed=42):
        self.n_agents = n_agents
        self.n_iter   = n_iter
        self.n_types  = n_types
        self.alpha    = alpha
        self.beta     = beta
        self.m_low    = m_low
        self.m_high   = m_high
        self.seed     = seed
        self.visit_counts_ = None

    def _normalize(self, X):
        mn = X.min(axis=0); mx = X.max(axis=0)
        return (X - mn) / (mx - mn + 1e-8)

    def select(self, features: np.ndarray, n_select: int) -> tuple:
        """
        features  : (N_pool, D) 도메인 피처 행렬
        n_select  : 선택할 예시 수
        returns   : (selected_indices, visit_counts)
        """
        rng = np.random.RandomState(self.seed)
        N_pool, D = features.shape
        K = self.n_types if self.n_types is not None else D
        X_norm = self._normalize(features)

        # 초기화
        agent_pos = X_norm[rng.choice(N_pool, self.n_agents, replace=True)].copy()
        agent_W   = rng.dirichlet(np.ones(K), self.n_agents)
        M         = rng.uniform(self.m_low, self.m_high, (K, K))
        visit     = np.zeros(N_pool)

        for _ in range(self.n_iter):
            C = agent_W @ M @ agent_W.T
            np.fill_diagonal(C, 0)

            for i in range(self.n_agents):
                ci = C[i].copy()
                n_nb = min(3, self.n_agents - 1)
                attract_idx = np.argsort(ci)[-n_nb:]
                repel_idx   = np.argsort(ci)[:n_nb]

                force = np.zeros(D, dtype=float)
                for j in attract_idx:
                    force += ci[j] * (agent_pos[j] - agent_pos[i])
                for j in repel_idx:
                    force += ci[j] * (agent_pos[j] - agent_pos[i])
                force /= (2 * n_nb)

                candidate = np.clip(agent_pos[i] + self.alpha * force, 0, 1)
                dists   = np.linalg.norm(X_norm - candidate, axis=1)
                nearest = int(np.argmin(dists))

                agent_pos[i] = X_norm[nearest]
                visit[nearest] += 1.0

                best_j = int(attract_idx[np.argmax(ci[attract_idx])])
                W_new  = (1 - self.beta) * agent_W[i] + self.beta * agent_W[best_j]
                agent_W[i] = W_new / W_new.sum()

        probs = visit + 1.0
        probs /= probs.sum()
        selected = np.random.RandomState(self.seed).choice(N_pool, n_select,
                                                            replace=False, p=probs)
        self.visit_counts_ = visit
        return selected, visit

print("AISOfewshot 정의 완료")

---
## 4. Global 선택 — fewshot_config.json 교체용

In [ ]:
N_SELECT_GLOBAL = 8   # 몇 개를 선택할지 (현재 fewshot은 5개)

aiso_global = AISOfewshot(
    n_agents=25,
    n_iter=120,
    n_types=None,   # 피처 차원(8)과 동일
    alpha=0.25,
    beta=0.08,
    m_low=-0.8,
    m_high=2.0,
    seed=42
)

print(f"AISO 실행 중 (n_agents={aiso_global.n_agents}, n_iter={aiso_global.n_iter})...")
global_idx, visit = aiso_global.select(X_pool, N_SELECT_GLOBAL)
print("완료")

print(f"\n선택된 인덱스: {sorted(global_idx)}")
print(f"방문된 고유 항목 수: {(visit > 0).sum():,} / {len(work_pool):,}")

In [ ]:
# 선택된 예시 내용 출력
print("=" * 70)
print("[AISO Global] 선택된 예시들")
print("=" * 70)

selected_entries = [work_pool[i] for i in global_idx]

for rank, (idx, entry) in enumerate(zip(global_idx, selected_entries)):
    feat = X_pool[idx]
    if feat[1] > 0:   pat = "REP"
    elif feat[2] > 0: pat = "DEN"
    elif feat[3] > 0: pat = "MEM"
    elif feat[4] > 0: pat = "ROY"
    else:             pat = "GEN"
    
    orig_len = len(entry["original"])
    if orig_len < 50:    lbucket = "XS"
    elif orig_len < 150: lbucket = "S"
    elif orig_len < 350: lbucket = "M"
    else:                lbucket = "L"
    
    print(f"\n[{rank+1}] ID={entry['id']}  패턴={pat}  길이={lbucket}({orig_len}자)")
    print(f"  원문: {entry['original'][:80]}{'...' if len(entry['original'])>80 else ''}")
    print(f"  번역: {entry['translation'][:100]}{'...' if len(entry['translation'])>100 else ''}")

In [ ]:
# 현재 고정 예시 vs AISO 선택 패턴 커버리지 비교
current_examples_patterns = ["REP", "REP", "REP", "REP", "REP"]  # fewshot_config.json 전부 REP

aiso_patterns = []
for idx in global_idx:
    feat = X_pool[idx]
    if feat[1] > 0:   aiso_patterns.append("REP")
    elif feat[2] > 0: aiso_patterns.append("DEN")
    elif feat[3] > 0: aiso_patterns.append("MEM")
    elif feat[4] > 0: aiso_patterns.append("ROY")
    else:             aiso_patterns.append("GEN")

print("패턴 커버리지 비교")
print(f"현재 고정 예시: {Counter(current_examples_patterns)}")
print(f"AISO 선택:     {Counter(aiso_patterns)}")

aiso_lengths = []
for idx in global_idx:
    n = len(work_pool[idx]["original"])
    if n < 50:    aiso_lengths.append("XS")
    elif n < 150: aiso_lengths.append("S")
    elif n < 350: aiso_lengths.append("M")
    else:         aiso_lengths.append("L")
print(f"AISO 길이 분포: {Counter(aiso_lengths)}")

In [ ]:
# fewshot_config.json 업데이트 저장
with open(BASE / "fewshot_config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

# 번역문만 예시로 사용 (현재 포맷 유지)
aiso_examples = [work_pool[i]["translation"] for i in global_idx]

config_aiso = {
    "fewshot_examples": aiso_examples,
    "prompt_template": config["prompt_template"],
    "meta": {
        "method": "AISO-global",
        "n_select": N_SELECT_GLOBAL,
        "patterns": aiso_patterns,
        "lengths": aiso_lengths,
        "source_ids": [work_pool[i]["id"] for i in global_idx]
    }
}

out_path = BASE / "fewshot_config_aiso_global.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(config_aiso, f, ensure_ascii=False, indent=2)
print(f"저장: {out_path}")
print("\n→ run_fewshot.py에서 fewshot_config.json 대신 이 파일을 사용하면 됩니다.")

---
## 5. Per-Query 동적 선택

각 쿼리(한문 원문)의 패턴과 길이를 파악한 뒤:
1. **앵커 예시** 1개: 쿼리와 같은 패턴 + 비슷한 길이 버킷
2. **다양성 예시** (N-1)개: AISO visit 상위 풀에서 앵커와 다른 패턴 우선

이 방식이 효과적인 이유: REP 쿼리에는 REP 어투 참조가 1개 필요하고, 나머지는 다양한 패턴으로 채워야 편향 없이 번역 가능.

In [ ]:
def get_pattern(feat: np.ndarray) -> str:
    if feat[1] > 0:   return "REP"
    elif feat[2] > 0: return "DEN"
    elif feat[3] > 0: return "MEM"
    elif feat[4] > 0: return "ROY"
    else:             return "GEN"

def get_length_bucket(orig: str) -> str:
    n = len(orig)
    if n < 50:    return "XS"
    elif n < 150: return "S"
    elif n < 350: return "M"
    else:         return "L"

# AISO 방문 상위 500개를 후보 풀로 축소 (속도)
TOP_CANDIDATES = 500
top_idx = np.argsort(visit)[::-1][:TOP_CANDIDATES]
top_entries  = [work_pool[i] for i in top_idx]
top_features = X_pool[top_idx]
top_patterns = [get_pattern(top_features[i]) for i in range(len(top_idx))]
top_lengths  = [get_length_bucket(e["original"]) for e in top_entries]

print(f"후보 풀 상위 {TOP_CANDIDATES}개 패턴 분포:")
print(Counter(top_patterns))

def select_perquery(query_original: str, n_select: int = 5, 
                    n_anchor: int = 1, rng_seed: int = 0) -> list:
    """
    단일 쿼리에 대해 동적으로 Few-Shot 예시 선택.

    Parameters
    ----------
    query_original : 번역할 한문 원문
    n_select       : 선택할 예시 총 수 (기본 5)
    n_anchor       : 쿼리와 같은 패턴 예시 수 (기본 1)

    Returns
    -------
    list of dict: 선택된 entry 리스트
    """
    rng = np.random.RandomState(rng_seed)
    q_feat   = extract_features({"original": query_original})
    q_pat    = get_pattern(q_feat)
    q_lbuck  = get_length_bucket(query_original)

    # 1) 앵커: 같은 패턴, 비슷한 길이 버킷 우선
    length_order = {"XS": 0, "S": 1, "M": 2, "L": 3}
    q_lval = length_order[q_lbuck]

    anchor_candidates = [
        (i, abs(length_order[top_lengths[i]] - q_lval))
        for i in range(len(top_entries))
        if top_patterns[i] == q_pat
    ]
    anchor_candidates.sort(key=lambda x: x[1])

    if not anchor_candidates:
        anchor_candidates = [(i, 0) for i in range(len(top_entries))]

    anchor_indices = [anchor_candidates[k][0] for k in range(min(n_anchor, len(anchor_candidates)))]
    used = set(anchor_indices)

    # 2) 다양성 예시: 앵커와 다른 패턴 우선
    diverse_candidates = [
        i for i in range(len(top_entries))
        if i not in used and top_patterns[i] != q_pat
    ]
    # 패턴 커버리지 극대화: 패턴별로 하나씩
    pat_seen = set(top_patterns[i] for i in anchor_indices)
    diverse_selected = []
    for target_pat in ["GEN", "REP", "DEN", "MEM", "ROY"]:
        if target_pat in pat_seen:
            continue
        for i in diverse_candidates:
            if top_patterns[i] == target_pat and i not in used:
                diverse_selected.append(i)
                used.add(i)
                pat_seen.add(target_pat)
                break
        if len(diverse_selected) + len(anchor_indices) >= n_select:
            break

    # 부족하면 남은 후보에서 채움
    remaining = [i for i in diverse_candidates if i not in used]
    rng.shuffle(remaining)
    while len(diverse_selected) + len(anchor_indices) < n_select and remaining:
        diverse_selected.append(remaining.pop(0))

    final_indices = anchor_indices + diverse_selected[:n_select - len(anchor_indices)]
    return [top_entries[i] for i in final_indices]

print("select_perquery 함수 정의 완료")

In [ ]:
# Per-query 선택 예시 시연
# eval300에서 다양한 패턴의 쿼리 3개 선택

# eval300 항목 로드 (원문 포함)
eval_entries = [e for e in corpus if e["id"] in eval_ids and e.get("original", "").strip()]
print(f"eval300 로드: {len(eval_entries)}개")

# 패턴별로 예시 쿼리 선택
demo_queries = {}
for e in eval_entries:
    f = extract_features(e)
    p = get_pattern(f)
    if p not in demo_queries:
        demo_queries[p] = e
    if len(demo_queries) >= 4:
        break

for pat, query_entry in demo_queries.items():
    print(f"\n{'='*65}")
    print(f"쿼리 패턴: {pat} | 길이: {len(query_entry['original'])}자")
    print(f"원문: {query_entry['original'][:80]}...")
    print(f"{'─'*65}")
    
    selected = select_perquery(query_entry["original"], n_select=5)
    for rank, sel in enumerate(selected):
        sel_feat = extract_features(sel)
        sel_pat  = get_pattern(sel_feat)
        print(f"  [{rank+1}] {sel_pat} | {len(sel['original'])}자 | {sel['translation'][:70]}...")

---
## 6. run_fewshot.py 통합 코드

기존 `run_fewshot.py`에 아래 코드를 추가하면 per-query 동적 선택을 사용할 수 있습니다.

In [ ]:
# ─── run_fewshot.py 통합용 헬퍼 ───────────────────────────────
# 이 코드를 run_fewshot.py에 붙여넣거나 import해서 사용

INTEGRATION_CODE = '''
# ─── AISO Few-Shot 동적 선택 통합 ────────────────────────────
# aiso_fewshot_selector.ipynb에서 export된 헬퍼 함수

import json, re
import numpy as np
from pathlib import Path

# 사전 계산된 AISO 결과 로드 (이 노트북 실행 후 생성)
with open("fewshot_config_aiso_global.json", encoding="utf-8") as f:
    AISO_GLOBAL_CONFIG = json.load(f)

# ── Mode A: Global (단순 교체) ─────────────────────────────────
def build_prompt_global(text: str, template: str) -> str:
    examples_str = "\\n".join(
        f"- {ex}" for ex in AISO_GLOBAL_CONFIG["fewshot_examples"]
    )
    return template.format(examples=examples_str, text=text)

# ── Mode B: Per-Query (동적) ───────────────────────────────────
# 아래 함수는 이 노트북에서 정의한 select_perquery를 사용
# 노트북을 한 번 실행한 뒤 top_entries, top_patterns 등이 메모리에 있어야 함

def build_prompt_perquery(text: str, template: str) -> str:
    selected = select_perquery(text, n_select=5)
    examples_str = "\\n".join(f"- {e[\'translation\']}" for e in selected)
    return template.format(examples=examples_str, text=text)

# ── 기존 run_fewshot.py 변경 지점 ─────────────────────────────
# 기존:
#   prompt = template.format(examples=fixed_examples_str, text=item["original"])
# AISO Global로 교체:
#   prompt = build_prompt_global(item["original"], template)
# AISO Per-Query로 교체:
#   prompt = build_prompt_perquery(item["original"], template)
'''

print(INTEGRATION_CODE)

# 통합 코드 파일로 저장
with open(BASE / "aiso_integration.py", "w", encoding="utf-8") as f:
    f.write(INTEGRATION_CODE.strip())
print("\n→ aiso_integration.py 저장 완료")

---
## 7. 다양성 시각화

In [ ]:
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 시각화용 서브샘플
viz_n = min(3000, len(X_pool))
viz_idx = np.random.choice(len(X_pool), viz_n, replace=False)
X_viz = X_pool[viz_idx]

pca = PCA(n_components=2, random_state=42)
X_all_2d = pca.fit_transform(X_pool)
X_viz_2d = X_all_2d[viz_idx]

# 현재 고정 예시 (REP 편향)
# fewshot_config.json의 5개는 모두 REP, 임의로 pool에서 REP 5개 선택
rep_indices = [i for i, p in enumerate(pattern_labels) if p == "REP(啓曰)"][:5]

pat_color = {"REP(啓曰)": "#3498db", "GEN(일반)": "#95a5a6",
             "DEN(不許)": "#e74c3c", "MEM(狀啓)": "#2ecc71", "ROY(上曰)": "#f39c12"}
colors_viz = [pat_color.get(pattern_labels[i], "#95a5a6") for i in viz_idx]

# 왼쪽: 현재 고정 예시
axes[0].scatter(X_viz_2d[:, 0], X_viz_2d[:, 1], c=colors_viz, s=8, alpha=0.4)
if rep_indices:
    axes[0].scatter(X_all_2d[rep_indices, 0], X_all_2d[rep_indices, 1],
                    c='#e74c3c', s=200, zorder=5, edgecolors='k', linewidths=1.5,
                    marker='*', label='현재 고정 예시(REP 편향)')
axes[0].set_title("현재 fewshot_config.json\n(5개 전부 REP 패턴)", fontsize=11)
axes[0].legend(fontsize=8)

# 오른쪽: AISO 선택
axes[1].scatter(X_viz_2d[:, 0], X_viz_2d[:, 1], c=colors_viz, s=8, alpha=0.4)
axes[1].scatter(X_all_2d[list(global_idx), 0], X_all_2d[list(global_idx), 1],
                c='#e74c3c', s=200, zorder=5, edgecolors='k', linewidths=1.5,
                marker='*', label=f'AISO 선택({N_SELECT_GLOBAL}개)')
for k, gi in enumerate(global_idx):
    axes[1].annotate(aiso_patterns[k], (X_all_2d[gi, 0], X_all_2d[gi, 1]),
                     fontsize=7, ha='left', va='bottom')
axes[1].set_title(f"AISO 선택 ({N_SELECT_GLOBAL}개)\n패턴 다양성 확보", fontsize=11)
axes[1].legend(fontsize=8)

# 범례 패치
import matplotlib.patches as mpatches
patches = [mpatches.Patch(color=c, label=p) for p, c in pat_color.items()]
fig.legend(handles=patches, loc='lower center', ncol=5, fontsize=8, bbox_to_anchor=(0.5, -0.05))

for ax in axes:
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(BASE / "aiso_fewshot_diversity.png", dpi=120, bbox_inches='tight')
plt.show()
print("→ aiso_fewshot_diversity.png 저장")

---
## 8. 체크리스트

### Global 모드 사용 순서
```
1. 이 노트북 전체 실행
2. fewshot_config_aiso_global.json 확인
3. run_fewshot.py에서 config 파일 경로를 교체
4. python run_fewshot.py
5. python score_300.py 로 BLEU/chrF/NER 비교
```

### Per-Query 모드 사용 순서
```
1. 이 노트북 전체 실행 (top_entries, select_perquery 메모리 필요)
2. run_fewshot.py 상단에 select_perquery 함수 붙여넣기
3. 프롬프트 생성 부분을 build_prompt_perquery로 교체
4. python run_fewshot.py
```

### 파라미터 튜닝 포인트
| 파라미터 | 현재값 | 바꿀 때 |
|----------|--------|----------|
| `N_SELECT_GLOBAL` | 8 | 6이나 10으로 실험 |
| `n_anchor` (per-query) | 1 | 2로 올리면 패턴 정확도 ↑, 다양성 ↓ |
| `TOP_CANDIDATES` | 500 | 작을수록 빠름, 클수록 커버리지 ↑ |
| `aiso.m_low` | -0.8 | 더 음수로 → repulsion 강도 ↑ |